<a href="https://colab.research.google.com/github/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
from wordcloud import WordCloud
import seaborn as sns

In [ ]:
import json
import logging
import random
import re
import time
from datetime import datetime
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry



In [ ]:
BASE_URL = "https://www.tovima.gr/category/texnologia/"
MAX_PAGES = 6
TARGET_YEAR = 2026
MIN_DELAY = 1.5
MAX_DELAY = 3.0
REQUEST_TIMEOUT = 15
SOURCE_NAME = "Το Βήμα"
OUTPUT_CSV = "tovima_texnologia_2026.csv"

In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    ),
    "Accept-Language": "el-GR,el;q=0.9,en-US;q=0.8,en;q=0.7",
}

STOP_PHRASES = [
    "Ακολούθησε το Βήμα στο",
    "Σχόλια",
    "Περισσότερα από ΤΟ ΒΗΜΑ",
    "Άφησε το σχόλιό σου",
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("tovima_scraper")

def build_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(HEADERS)
    retries = Retry(
        total=3,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.mount("http://", HTTPAdapter(max_retries=retries))
    return session

In [ ]:
DATE_PATTERN = re.compile(r"(\d{1,2})\.(\d{1,2})\.(\d{4})(?:,?\s*(\d{1,2}):(\d{2}))?")


def parse_date_from_text(text: str):
    """Ψάχνει μοτίβο DD.MM.YYYY[, HH:MM] μέσα σε ελεύθερο κείμενο."""
    if not text:
        return None
    match = DATE_PATTERN.search(text)
    if not match:
        return None
    day, month, year, hour, minute = match.groups()
    try:
        if hour and minute:
            return datetime(int(year), int(month), int(day), int(hour), int(minute))
        return datetime(int(year), int(month), int(day))
    except ValueError:
        return None

In [ ]:
def extract_article(story, base_url: str):
    story_text = story.get_text(" ", strip=True)
    parsed_date = parse_date_from_text(story_text)

    if parsed_date is not None and parsed_date.year != TARGET_YEAR:
        return None
    if parsed_date is None and str(TARGET_YEAR) not in story_text:
        return None

    headline_tag = story.find("h3")
    headline = headline_tag.get_text(" ", strip=True) if headline_tag else None

    link_tag = None
    if headline_tag is not None:
        link_tag = headline_tag.find("a", href=True)
    if link_tag is None:
        link_tag = story.find("a", href=True)
    article_url = urljoin(base_url, link_tag["href"]) if link_tag else None

    author_tag = story.find("span", class_="vima-author")
    author = author_tag.get_text(" ", strip=True) if author_tag else None
    return {
        "site": SOURCE_NAME,
        "url": article_url,
        "title": headline,
        "date": parsed_date.date() if parsed_date else None,
        "datetime": parsed_date,
        "author": author,
    }

In [ ]:
def extract_full_text(session: requests.Session, url: str) -> str | None:
    if not url:
        return None
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
    except requests.RequestException as exc:
        log.warning("Αποτυχία fetch άρθρου (%s): %s", url, exc)
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # 1η προσπάθεια: <article> tag (τυπικό σε WordPress themes)
    container = soup.find("article")

    # 2η προσπάθεια: γνωστά ονόματα container για article body
    if container is None:
        container = soup.find(
            "div",
            class_=re.compile(
                r"(entry-content|single-content|article-content|post-content|content-single)",
                re.IGNORECASE,
            ),
        )

    # Fallback: όλη η σελίδα (λιγότερο ακριβές, αλλά καλύτερο από τίποτα)
    search_root = container if container is not None else soup

    paragraphs = search_root.find_all("p")

    text_parts = []
    for p in paragraphs:
        text = p.get_text(" ", strip=True)
        if not text:
            continue
        # Σταματάμε μόλις φτάσουμε σε boilerplate/σχόλια/related content
        if any(stop in text for stop in STOP_PHRASES):
            break
        text_parts.append(text)

    full_text = "\n\n".join(text_parts).strip()
    return full_text if full_text else None


In [ ]:
def scrape_metadata() -> pd.DataFrame:
    session = build_session()
    articles_list = []

    for page in range(1, MAX_PAGES + 1):
        url = BASE_URL if page == 1 else f"{BASE_URL}page/{page}/"
        log.info("Σελίδα %d: %s", page, url)

        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
        except requests.RequestException as exc:
            log.warning("Αποτυχία στη σελίδα %d (%s) — προσπερνάω.", page, exc)
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        stories = soup.find_all("div", class_="wrap-category-row")
        log.info("Βρέθηκαν %d article blocks", len(stories))

        if not stories:
            log.info("Καμία ανάρτηση στη σελίδα %d — σταματάω το pagination.", page)
            break

        for story in stories:
            try:
                article = extract_article(story, url)
            except Exception as exc:
                log.warning("Σφάλμα στην εξαγωγή άρθρου: %s", exc)
                continue
            if article is not None:
                articles_list.append(article)

        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    df = pd.DataFrame(articles_list)
    if not df.empty:
        df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    return df



In [ ]:
def enrich_with_full_text(df: pd.DataFrame) -> pd.DataFrame:
    session = build_session()
    full_texts = []

    total = len(df)
    for i, url in enumerate(df["url"], start=1):
        log.info("Άρθρο %d/%d: %s", i, total, url)
        text = extract_full_text(session, url)
        full_texts.append(text)
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    df = df.copy()
    df["full_text"] = full_texts
    # Τελική σειρά στηλών
    df = df[["site", "url", "title", "date", "author", "full_text", "datetime"]]
    return df


def save_outputs(df: pd.DataFrame):
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    log.info("Αποθηκεύτηκε: %s (%d γραμμές)", OUTPUT_CSV, len(df))


def scrape() -> pd.DataFrame:
    """Πλήρης ροή: metadata + full text, σε ένα κλείσιμο."""
    df_meta = scrape_metadata()
    if df_meta.empty:
        log.warning("Δεν βρέθηκαν άρθρα — έλεγξε τα selectors (class names) του site.")
        return df_meta
    log.info("Μάζεψα %d άρθρα. Τώρα κατεβάζω το πλήρες κείμενο ενός-ενός...", len(df_meta))
    df_full = enrich_with_full_text(df_meta)
    return df_full


if __name__ == "__main__":
    df_tovima = scrape()
    log.info("Συνολικά μοναδικά άρθρα του %d: %d", TARGET_YEAR, len(df_tovima))
    if not df_tovima.empty:
        print(df_tovima.head(5).to_string())
        save_outputs(df_tovima)


      site                                                                                                                                          url                                                                                 title        date              author                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
import base64
import requests
from google.colab import userdata

def save_df_to_github(df, repo, path, token=None, branch="main", message="Update dataset"):
    token = token or userdata.get("GITHUB_TOKEN")
    csv_content = df.to_csv(index=False, encoding="utf-8-sig")
    content_b64 = base64.b64encode(csv_content.encode("utf-8-sig")).decode("utf-8")

    url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github+json"}

    existing = requests.get(url, headers=headers, params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None

    payload = {"message": message, "content": content_b64, "branch": branch}
    if sha:
        payload["sha"] = sha

    response = requests.put(url, headers=headers, json=payload)
    response.raise_for_status()
    print(f"✅ Αποθηκεύτηκε: https://github.com/{repo}/blob/{branch}/{path}")
    return response.json()

save_df_to_github(
    df_tovima,
    repo="annatsamoyra-prog/data-story-",
    path="tovima_texnologia_2026.csv",
    token=userdata.get("newtoken")
)

✅ Αποθηκεύτηκε: https://github.com/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia_2026.csv


{'content': {'name': 'tovima_texnologia_2026.csv',
  'path': 'tovima_texnologia_2026.csv',
  'sha': '4e595a4c16d7b164fa545eef7a5dcc69d932573e',
  'size': 16778907,
  'url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/tovima_texnologia_2026.csv?ref=main',
  'html_url': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia_2026.csv',
  'git_url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/4e595a4c16d7b164fa545eef7a5dcc69d932573e',
  'download_url': 'https://raw.githubusercontent.com/annatsamoyra-prog/data-story-/main/tovima_texnologia_2026.csv',
  'type': 'file',
  '_links': {'self': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/tovima_texnologia_2026.csv?ref=main',
   'git': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/4e595a4c16d7b164fa545eef7a5dcc69d932573e',
   'html': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/tovima_texnologia_2026.csv'}},
 'c

In [ ]:
len(df_tovima)

112